# EDA — Nomeroff OCR RU dataset

Разведочный анализ выборки русских автомобильных номеров (AUTO.RIA Numberplate OCR RU).
Цель — понять структуру данных перед оценкой OCR: длины номеров, алфавит символов,
размеры изображений, примеры кадров и потенциальные сложности (мелкие/смазанные кропы).

Запуск: ядро окружения `anpr`, рабочая директория — `project/`.

In [ ]:
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# allow `import src...` when running from project/notebooks
sys.path.append(str(Path.cwd().parent))
from src.data.nomeroff import load_samples
from src.postprocess import normalize_plate

# Adjust to where the dataset is unpacked on this machine:
DATA_DIR = r"C:\Users\USER\projects\datasets\autoriaNumberplateOcrRu-2021-09-01"
SPLIT = "test"

In [ ]:
samples = load_samples(DATA_DIR, split=SPLIT, limit=None)
print(f"Split '{SPLIT}': {len(samples)} samples")
samples[:5]

## 1. Распределение длины номера (число символов после нормализации)

In [ ]:
lengths = [len(normalize_plate(s.text)) for s in samples]
len_counter = Counter(lengths)
print(sorted(len_counter.items()))

plt.figure(figsize=(7, 4))
plt.bar(list(len_counter.keys()), list(len_counter.values()))
plt.xlabel("Длина номера (символов)")
plt.ylabel("Количество")
plt.title("Распределение длины номеров")
plt.show()

## 2. Частотность символов в алфавите номеров

In [ ]:
char_counter = Counter()
for s in samples:
    char_counter.update(normalize_plate(s.text))

chars = sorted(char_counter.keys())
counts = [char_counter[c] for c in chars]
print("Уникальных символов:", len(chars))
print("Алфавит:", "".join(chars))

plt.figure(figsize=(12, 4))
plt.bar(chars, counts)
plt.xlabel("Символ")
plt.ylabel("Частота")
plt.title("Частотность символов")
plt.show()

## 3. Размеры изображений (кропов номеров)

In [ ]:
import random

subset = random.sample(samples, min(1000, len(samples)))
sizes = np.array([Image.open(s.image_path).size for s in subset])  # (W, H)
widths, heights = sizes[:, 0], sizes[:, 1]
print(f"Ширина:  min={widths.min()} max={widths.max()} median={np.median(widths):.0f}")
print(f"Высота:  min={heights.min()} max={heights.max()} median={np.median(heights):.0f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(widths, bins=30); ax[0].set_title("Ширина, px")
ax[1].hist(heights, bins=30); ax[1].set_title("Высота, px")
plt.show()

Мелкие кропы (малая высота) — основная причина ошибок OCR. Это мотивирует шаг
предобработки `improved` (апскейл до 200px + CLAHE) в `src/models/ocr.py`.

## 4. Примеры кадров

In [ ]:
examples = random.sample(samples, 12)
fig, axes = plt.subplots(3, 4, figsize=(14, 6))
for ax, s in zip(axes.ravel(), examples):
    ax.imshow(Image.open(s.image_path))
    ax.set_title(normalize_plate(s.text), fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Выводы

_Заполняется после запуска на машине с датасетом:_

- типичная длина номера: …
- алфавит: латинские эквиваленты RU-букв (ABCEHKMOPTXY) + цифры;
- разброс размеров кропов: … — мелкие требуют апскейла;
- наблюдения по качеству (смаз, наклон, грязь): …